# 03 - LLM chat NER (Task 4)
A chat LLM (no API) extracts the entities of the first 5 documents. The answer is converted to the
same format as GLiNER (offsets are found by searching the entity text) and evaluated with strict
and relaxed matching.

In [1]:
import os, sys, warnings
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path('..').resolve()))

%load_ext autoreload
%autoreload 2

In [2]:
from ner.evaluation import evaluate_ner_strict, evaluate_ner_relaxed, ENTITY_LABELS
from ner.llm import build_prompt, load_llm_answer, llm_records_to_responses
from ner.utils import load_jsonl, save_responses_to_jsonl

VALIDATION_PATH = '../data/plain/validation.jsonl'

LABELS = list(ENTITY_LABELS)

N_DOCS = 5
llm_gold = load_jsonl(VALIDATION_PATH, n=N_DOCS)          # los 5 primeros documentos
print(len(llm_gold), 'documents for the LLM')

5 documents for the LLM


In [3]:
prompt = build_prompt(llm_gold, LABELS)

PROMPT_PATH = Path('../results/llm/prompt.txt')
PROMPT_PATH.write_text(prompt, encoding='utf-8')

print('Prompt length:', len(prompt), 'characters')
print('Saved in', PROMPT_PATH)
print('\n', prompt[:1200])

Prompt length: 19864 characters
Saved in ..\results\llm\prompt.txt

 You are a Named Entity Recognition system for administrative documents written in Spanish or Catalan.
Extract ALL the entities of these types: PERSON_NAME, ID_CARD_NUMBER, ADDRESS, REGISTRATION_NUMBER, DATE, AMOUNT, ORGANIZATION.
Rules:
- Copy the entity text EXACTLY as it appears in the document (same characters, accents and punctuation).
- If an entity appears several times in a document, list it once for EACH appearance, in order.
- ADDRESS is the full address as one entity (street, number, floor/door, postal code and city).
- ORGANIZATION is the full name of the public body (e.g. "Servicio de ... de la Región Central").
- AMOUNT includes the currency (e.g. "4.416,13 euros", "1.107,16 EUR", "5.523,29 €").
- Answer ONLY with JSON Lines: one line per document, no explanations, no markdown.
- Format of each line: {"id": "<id>", "entities": [{"text": "...", "label": "..."}]}

Documents:
[id=test-0001]
Servicio de Proce

In [4]:
def evaluate_llm(llm_name, answer_path):
    """Read an LLM chat answer, convert it to the GLiNER format, save it and evaluate it."""
    # --- 1. Leer la respuesta copiada del chat ---
    llm_records = load_llm_answer(answer_path)
    print(f'{llm_name}: {len(llm_records)} documents in the answer')

    # --- 2. Calcular start/end buscando cada entidad en el texto original ---
    responses = llm_records_to_responses(llm_records, llm_gold)
    pred_path = f'../results/llm/{llm_name}_predictions.jsonl'
    save_responses_to_jsonl(responses, pred_path)

    # --- 3. Entidades que el LLM escribió distinto (no aparecen tal cual en el texto) ---
    not_found = [(r['id'], e['label'], e['text'])
                 for r in responses for e in r['entities'] if e['start'] == -1]
    print(f'{len(not_found)} entities not found literally in the text:')
    for item in not_found:
        print('   ', item)

    # --- 4. Evaluar ---
    print(f'\n########## {llm_name} ##########')
    evaluate_ner_strict(VALIDATION_PATH, pred_path)
    evaluate_ner_relaxed(VALIDATION_PATH, pred_path);

In [5]:
evaluate_llm('chatgpt', '../results/llm/llm_raw_answer_chatgpt.txt')

chatgpt: 5 documents in the answer
0 entities not found literally in the text:

########## chatgpt ##########

STRICT EVALUATION
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 205 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00      50       0       0
ID_CARD_NUMBER              1.00   1.00   1.00      20       0       0
ADDRESS                     1.00   1.00   1.00      20       0       0
REGISTRATION_NUMBER         1.00   1.00   1.00      25       0       0
DATE                        1.00   1.00   1.00      40       0       0
AMOUNT                      1.00   1.00   1.00      20       0       0
ORGANIZATION                1.00   1.00   1.00      30       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 205 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1

In [6]:
evaluate_llm('gemini', '../results/llm/llm_raw_answer_gemini.txt')

gemini: 5 documents in the answer
0 entities not found literally in the text:

########## gemini ##########

STRICT EVALUATION
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 205 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00      50       0       0
ID_CARD_NUMBER              1.00   1.00   1.00      20       0       0
ADDRESS                     1.00   1.00   1.00      20       0       0
REGISTRATION_NUMBER         1.00   1.00   1.00      25       0       0
DATE                        1.00   1.00   1.00      40       0       0
AMOUNT                      1.00   1.00   1.00      20       0       0
ORGANIZATION                1.00   1.00   1.00      30       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 205 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1  

In [7]:
gliner5 = load_jsonl('../results/plain/gliner_small_merge.jsonl', n=N_DOCS)
GLINER5_PATH = '../results/llm/gliner_small_merge_first5_reference.jsonl'
save_responses_to_jsonl(gliner5, GLINER5_PATH)

print('########## GLiNER small + merge (same 5 documents) ##########')
evaluate_ner_strict(VALIDATION_PATH, GLINER5_PATH)
evaluate_ner_relaxed(VALIDATION_PATH, GLINER5_PATH);

########## GLiNER small + merge (same 5 documents) ##########

STRICT EVALUATION
Micro -> P: 0.91 | R: 0.94 | F1: 0.93 | TP: 193 | FP: 18 | FN: 12

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.98   1.00   0.99      50       1       0
ID_CARD_NUMBER              0.89   0.85   0.87      17       2       3
ADDRESS                     0.90   0.90   0.90      18       2       2
REGISTRATION_NUMBER         1.00   0.92   0.96      23       0       2
DATE                        1.00   1.00   1.00      40       0       0
AMOUNT                      0.67   0.80   0.73      16       8       4
ORGANIZATION                0.85   0.97   0.91      29       5       1

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.91 | R: 0.94 | F1: 0.93 | TP: 193 | FP: 18 | FN: 12

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------

### Task 4 — Results (first 5 documents)

LLMs used through their chat interface (no API): **ChatGPT (model: ...)** and **Gemini (model: ...)**.
The same prompt (`results/llm/prompt.txt`) was given to both, in a new chat.

| System | Strict F1 | Relaxed occ. F1 | Relaxed unique F1 |
|---|---|---|---|
| GLiNER small + merge | 0.93 | 0.93 | 0.91 |
| ChatGPT | 1.00 | 1.00 | 1.00 |
| Gemini | 1.00 | 1.00 | 1.00 |

- **Both LLMs got a perfect score** (205/205 entities) and copied every entity exactly as it appears
  in the text (0 entities not found), so the offsets could be computed by searching the text.
- GLiNER makes mistakes mainly in AMOUNT (0.73), ID_CARD_NUMBER (0.87) and ADDRESS/ORGANIZATION.
- **The comparison is not completely fair:**
  - only 5 documents were evaluated with the LLMs;
  - the prompt included **annotation rules** (full address, amount with currency, one entity per
    mention), while GLiNER only receives the label names. GLiNER's errors are exactly of this kind.
- **Practical limitations of the LLM chat approach:** manual copy/paste and length limits (not
  scalable to many documents), no character offsets, results depend on the model version, and
  **privacy**: real administrative documents (ID numbers, addresses) should not be sent to an
  external service. GLiNER is small, free, runs locally on CPU and gives offsets directly.
- The answers had copy/paste issues (Markdown escapes like `\_`, all documents on one line), solved
  with a robust parser (`ner/llm.py`).